In [1]:
# Importing necessary libraries
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score

### Define main function

In [4]:
def evaluate_knn_with_reduction(X, y,
                                 n_neighbors=5, n_component_pca=100, n_component_lda=9,
                                 test_size=0.3, random_state=42):
    """
    Đánh giá KNN với 4 cách xử lý chiều dữ liệu:
    1. Không giảm chiều
    2. PCA
    3. LDA
    4. PCA -> LDA

    Parameters
    ----------
    X : array-like
        Toàn bộ features (chưa chia train/test, chưa chuẩn hóa).
    y : array-like
        Toàn bộ nhãn tương ứng.
    n_neighbors : int
        Số neighbor cho KNN.
    n_component_pca : int
        Số chiều giữ lại khi dùng PCA.
    n_component_lda : int
        Số chiều giữ lại khi dùng LDA (tối đa = n_classes - 1).
    test_size : float
        Tỉ lệ tập test khi split.
    random_state : int
        Seed cho train_test_split.

    Returns
    -------
    dict với 4 giá trị accuracy:
        - "no_reduction"
        - "pca"
        - "lda"
        - "pca_lda"
    """

    # --- Bước 1: Chia train/test ---
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    # --- Bước 2: Chuẩn hóa dữ liệu ---
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # --- Hàm phụ: train KNN và trả về accuracy ---
    def _run_knn(X_tr, X_te):
        knn = KNeighborsClassifier(n_neighbors=n_neighbors)
        knn.fit(X_tr, y_train)
        y_pred = knn.predict(X_te)
        return accuracy_score(y_test, y_pred)

    results = {}

    # 1. Không giảm chiều
    results["no_reduction"] = _run_knn(X_train_scaled, X_test_scaled)

    # 2. PCA
    pca = PCA(n_components=n_component_pca)
    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca = pca.transform(X_test_scaled)
    results["pca"] = _run_knn(X_train_pca, X_test_pca)

    # 3. LDA
    lda = LinearDiscriminantAnalysis(n_components=n_component_lda)
    X_train_lda = lda.fit_transform(X_train_scaled, y_train)
    X_test_lda = lda.transform(X_test_scaled)
    results["lda"] = _run_knn(X_train_lda, X_test_lda)

    # 4. PCA -> LDA (dùng lại X_train_pca / X_test_pca ở bước 2)
    lda_on_pca = LinearDiscriminantAnalysis(n_components=n_component_lda)
    X_train_pca_lda = lda_on_pca.fit_transform(X_train_pca, y_train)
    X_test_pca_lda = lda_on_pca.transform(X_test_pca)
    results["pca_lda"] = _run_knn(X_train_pca_lda, X_test_pca_lda)

    return results

In [ ]:
# Load the MNIST dataset
mnist = fetch_openml('mnist_784', version=1)
mnist_feature = mnist.data.astype('float64')
mnist_target = mnist.target.astype('int64')

results = evaluate_knn_with_reduction(
    X = mnist_feature, y = mnist_target,
    n_neighbors=5, n_component_pca=100, n_component_lda=9
)

print("Accuracy of KNN without dimensionality reduction:", results["no_reduction"])
print("Accuracy of KNN with PCA:", results["pca"])
print("Accuracy of KNN with LDA:", results["lda"])
print("Accuracy of KNN with PCA(100) -> LDA(8):", results["pca_lda"])


Accuracy of KNN without dimensionality reduction: 0.943
Accuracy of KNN with PCA: 0.9566190476190476
Accuracy of KNN with LDA: 0.9136190476190477
Accuracy of KNN with PCA(8) -> LDA(1): 0.9184285714285715


In [9]:
#Load breast cancer dataset
data = load_breast_cancer()
cancer_feature = pd.DataFrame(data.data, columns=data.feature_names)
cancer_target = data.target

results = evaluate_knn_with_reduction(
    X = cancer_feature, y = cancer_target,
    n_neighbors=5, n_component_pca=8, n_component_lda=1
)

print("Accuracy of KNN without dimensionality reduction:", results["no_reduction"])
print("Accuracy of KNN with PCA:", results["pca"])
print("Accuracy of KNN with LDA:", results["lda"])
print("Accuracy of KNN with PCA(8) -> LDA(1):", results["pca_lda"])


Accuracy of KNN without dimensionality reduction: 0.9590643274853801
Accuracy of KNN with PCA: 0.9649122807017544
Accuracy of KNN with LDA: 0.9532163742690059
Accuracy of KNN with PCA(8) -> LDA(1): 0.9824561403508771
